In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 12.5 MB/s eta 0:00:00


In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import numpy as np
import json, re
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [ ]:

class InterviewDataset(Dataset):
    def __init__(self, master_csv, jd_csv):
        df = pd.read_csv(master_csv)
        jd = pd.read_csv(jd_csv)
        df = df.merge(jd[['jd_id','jd_text_embedding']], on='jd_id', how='left')

        def parse_emb(s):
            try:
                return np.array(json.loads(s))
            except:
                parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
                return np.array([float(x) for x in parts if x])

        def parse_skills(s):
            return np.array(json.loads(s.replace("'", '"')))

        # 1) Define the interviewer-rating columns
        rating_cols = [
            'Ratings.Technical_Proficiency',
            'Ratings.Problem_Solving_Ability',
            'Ratings.Communication_Skills',
            'Ratings.Cultural_Team_Fit',
            'Ratings.Adaptability_Learning'
        ]
        # 2) Ensure they're numeric
        df[rating_cols] = df[rating_cols].apply(pd.to_numeric, errors='coerce').fillna(0.)

        # 3) Build your struct vector as: [skills_vec, segment_count, *ratings_vec]
        df['struct'] = df.apply(lambda r: np.concatenate([
            parse_skills(r['skills_vector']),
            [r['segment_count']],
            r[rating_cols].values.astype(float)
        ]), axis=1)

        # the rest stays the same…
        df['text'] = df['transcript_embedding'].apply(parse_emb)
        df['jd']   = df['jd_text_embedding'].  apply(parse_emb)

        df = df.dropna(subset=['struct','text','jd'])
        self.s = np.vstack(df['struct'].values).astype(np.float32)
        self.t = np.vstack(df['text'].  values).astype(np.float32)
        self.j = np.vstack(df['jd'].    values).astype(np.float32)

        self.y_reg = df['Overall_Score'].values.astype(np.float32)
        rec_cols   = ['rec_Hire','rec_Consider','rec_Reject']
        self.y_cls = df[rec_cols].values.argmax(axis=1).astype(np.int64)

    def __len__(self):
        return len(self.y_reg)

    def __getitem__(self, idx):
        return (self.s[idx], self.j[idx], self.t[idx],
                self.y_reg[idx], self.y_cls[idx])


In [ ]:
# --- Cross‑Attention Fusion Model ---
class CrossAttentionFusion(nn.Module):
    def __init__(self, dim_s, dim_j, dim_t, hidden, heads, dropout):
        super().__init__()
        # Projections
        self.proj_s = nn.Sequential(nn.Linear(dim_s, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_j = nn.Sequential(nn.Linear(dim_j, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.proj_t = nn.Sequential(nn.Linear(dim_t, hidden), nn.ReLU(), nn.Dropout(dropout))
        # Self‑attention over the 3 modality tokens
        self.attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=heads, dropout=dropout, batch_first=True)
        # Prediction heads
        self.head_reg = nn.Linear(hidden, 1)
        self.head_cls = nn.Linear(hidden, 3)

    def forward(self, s, j, t):
        hs = self.proj_s(s).unsqueeze(1)  # [B,1,H]
        hj = self.proj_j(j).unsqueeze(1)
        ht = self.proj_t(t).unsqueeze(1)
        seq = torch.cat([hs, hj, ht], dim=1)  # [B,3,H]
        attn_out, _ = self.attn(seq, seq, seq)
        fused = attn_out.mean(dim=1)  # [B,H]
        return fused

    def predict(self, s, j, t):
        fused = self.forward(s, j, t)
        return self.head_reg(fused).squeeze(-1), self.head_cls(fused)

In [ ]:
# --- Optuna Objective for Cross‑Attention ---
def objective(trial):
    # Hyperparameters
    hidden      = trial.suggest_categorical('hidden', [64, 128, 256])
    heads       = trial.suggest_categorical('heads', [2, 4, 8])
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    lr          = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay= trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    lambda_ce   = trial.suggest_float('lambda_ce', 0.1, 10.0, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [16, 32])
    n_epochs    = 8

    # Data & CV
    dataset = InterviewDataset('FINAL_master.csv', 'jds_clean.csv')
    dim_s, dim_j, dim_t = dataset.s.shape[1], dataset.j.shape[1], dataset.t.shape[1]
    kf = KFold(n_splits=3, shuffle=True, random_state=42)

    combined_metrics = []
    for train_idx, val_idx in kf.split(dataset):
        train_ds = Subset(dataset, train_idx)
        val_ds   = Subset(dataset, val_idx)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

        model = CrossAttentionFusion(dim_s, dim_j, dim_t, hidden, heads, dropout)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        mse_loss = nn.MSELoss()
        ce_loss  = nn.CrossEntropyLoss()

        # Train
        for _ in range(n_epochs):
            model.train()
            for s,j,t,y_reg,y_cls in train_loader:
                s,j,t = s.to(device), j.to(device), t.to(device)
                y_reg, y_cls = y_reg.to(device), y_cls.to(device)
                optimizer.zero_grad()
                pr, pc = model.predict(s,j,t)
                loss = mse_loss(pr, y_reg) + lambda_ce * ce_loss(pc, y_cls)
                loss.backward()
                optimizer.step()

        # Validate
        model.eval()
        regs, preds_reg, cls_true, cls_pred = [], [], [], []
        with torch.no_grad():
            for s,j,t,y_reg,y_cls in val_loader:
                s,j,t = s.to(device), j.to(device), t.to(device)
                pr, pc = model.predict(s,j,t)
                regs.extend(y_reg.numpy())
                preds_reg.extend(pr.cpu().numpy())
                cls_true.extend(y_cls.numpy())
                cls_pred.extend(pc.argmax(dim=1).cpu().numpy())

        mse = mean_squared_error(regs, preds_reg)
        rmse = np.sqrt(mse)
        acc  = accuracy_score(cls_true, cls_pred)
        combined_metrics.append(acc + (1 - rmse))

    return np.mean(combined_metrics)


In [ ]:
if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30, timeout=3600)
    print("Best trial params:", study.best_trial.params)
    print("Best combined metric:", study.best_value)

[I 2025-04-23 06:19:43,494] A new study created in memory with name: no-name-cec4476a-b648-4c13-9347-709b41e33b7f
[I 2025-04-23 06:19:53,796] Trial 0 finished with value: 1.1048163735486847 and parameters: {'hidden': 64, 'heads': 2, 'dropout': 0.4003281096062059, 'lr': 9.418128785529236e-05, 'weight_decay': 1.4430737654175438e-06, 'lambda_ce': 4.552666293998457, 'batch_size': 16}. Best is trial 0 with value: 1.1048163735486847.
[I 2025-04-23 06:19:54,797] Trial 1 finished with value: 0.8316262646715513 and parameters: {'hidden': 64, 'heads': 2, 'dropout': 0.303180115306243, 'lr': 2.4121847401518745e-05, 'weight_decay': 0.0004037195998675397, 'lambda_ce': 0.14383380307632418, 'batch_size': 32}. Best is trial 0 with value: 1.1048163735486847.
[I 2025-04-23 06:19:56,314] Trial 2 finished with value: 1.228793321643325 and parameters: {'hidden': 64, 'heads': 4, 'dropout': 0.4570265560479385, 'lr': 0.0008331670094444144, 'weight_decay': 4.403201133615e-05, 'lambda_ce': 0.8025598720328246, 'b

Best trial params: {'hidden': 256, 'heads': 8, 'dropout': 0.2316164643593665, 'lr': 0.0009463826656390514, 'weight_decay': 1.576745604200717e-05, 'lambda_ce': 0.37633068643404705, 'batch_size': 16}
Best combined metric: 1.719480006956082


training



Best trial params: {'hidden': 256, 'heads': 2, 'dropout': 0.45195498928161315, 'lr': 0.0009747086765017714, 'weight_decay': 1.936002648196862e-06, 'lambda_ce': 0.6206561489112559, 'batch_size': 32}
Best combined metric: 1.3314125031493287


In [ ]:
# === Load, Split, and Train/Evaluate ===
master_csv = 'FINAL_master.csv'
jd_csv     = 'jds_clean.csv'
dataset    = InterviewDataset(master_csv, jd_csv)

In [ ]:
# Stratified 10% hold‑out
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.1, random_state=42, stratify=dataset.y_cls)
train_ds = Subset(dataset, train_idx)
test_ds  = Subset(dataset, test_idx)

In [ ]:
# Dataloaders
batch_size = 16  # Best from Optuna
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dim_s, dim_j, dim_t = dataset.s.shape[1], dataset.j.shape[1], dataset.t.shape[1]

In [ ]:
# Best hyperparameters
hidden       = 256
heads        = 8
dropout      = 0.2316164643593665
lr           = 0.0009463826656390514
weight_decay = 1.576745604200717e-05
lambda_ce    = 0.37633068643404705
n_epochs     = 30

model = CrossAttentionFusion(dim_s, dim_j, dim_t, hidden, heads, dropout).to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
mse_loss = nn.MSELoss()
ce_loss  = nn.CrossEntropyLoss()

In [ ]:
# Training loop
for epoch in range(1, n_epochs+1):
    model.train()
    for s,j,t,y_reg,y_cls in train_loader:
        s,j,t = s.to(device), j.to(device), t.to(device)
        y_reg,y_cls = y_reg.to(device), y_cls.to(device)
        opt.zero_grad()
        pr, pc = model.predict(s, j, t)
        loss = mse_loss(pr, y_reg) + lambda_ce * ce_loss(pc, y_cls)
        loss.backward()
        opt.step()

In [ ]:
model.eval()
y_true_reg, y_pred_reg, y_true_cls, y_pred_cls = [], [], [], []
with torch.no_grad():
    for s,j,t,y_reg,y_cls in test_loader:
        s,j,t = s.to(device), j.to(device), t.to(device)
        pr, pc = model.predict(s, j, t)
        y_true_reg.extend(y_reg.numpy())
        y_pred_reg.extend(pr.cpu().numpy())
        y_true_cls.extend(y_cls.numpy())
        y_pred_cls.extend(pc.argmax(dim=1).cpu().numpy())

test_rmse = mean_squared_error(y_true_reg, y_pred_reg)
test_rmse = np.sqrt(test_rmse)
test_acc  = accuracy_score(y_true_cls, y_pred_cls)
print(f"► Cross‑Attention Test RMSE: {test_rmse:.4f}")
print(f"► Cross‑Attention Test Accuracy: {test_acc:.4f}")

► Cross‑Attention Test RMSE: 0.0811
► Cross‑Attention Test Accuracy: 0.8710


In [ ]:
torch.save(model.state_dict(), "cross_attention_model.pth")